In [ ]:
import pandas as pd
import numpy as np
import json
from google.genai import types

# -------------------------------------------------
# LOAD SPACY MODEL FOR LINGUISTIC BACKUP
# -------------------------------------------------
# nlp = spacy.load("en_core_web_sm")

# -------------------------------------------------
# OPENAI / LLM CONFIG
# -------------------------------------------------
# openai.api_key = "YOUR_API_KEY"   # <--- replace

LLM_MODEL = "gpt-4o-mini"         # cheap + accurate for semantic labels


# -------------------------------------------------
# 30 LLM FEATURE DEFINITIONS (PROMPT-BASED)
# -------------------------------------------------

FEATURE_SCHEMA = {
    "police_presence_signals": {
        "police_identified_by_role",
        "police_commands_present",
        "id_request_event",
        "legal_procedure_language",
        "police_explaining_actions",
        "civilian_questioning_police"
    },
    "interaction_type_signals": {
        "conversation_present",
        "verbal_disagreement_detected",
        "compliance_discussion",
        "instruction_clarification_dialogue",
        "negotiation_attempts",
        "conflict_resolution_attempt"
    },
    "escalation_indicators": {
        "raised_threat_language",
        "noncompliance_resistance_language",
        "accusatory_language",
        "police_warning_language",
        "emotional_intensity_spike",
        "crowd_escalation_factors",
        "use_of_force_transition_signals"
    },
    "deescalation_indicators": {
        "calming_language_used_by_officer",
        "empathetic_statements",
        "clear_explanations_given",
        "tone_softening_cues",
        "conflict_deescalation_success",
        "agreement_reached"
    },
    "context_filters": {
        "no_police_presence",
        "no_conversation_detected",
        "advertisement_content_detected",
        "training_range_context",
        "non_relevant_crime_only_context"
    }
}


# -------------------------------------------------
# LLM CALL FOR HIGH-LEVEL SEMANTIC FEATURES
# -------------------------------------------------


# Ensure you have your client initialized somewhere
# client = genai.Client(api_key="YOUR_API_KEY")

def extract_llm_gemini_features(text):
    """
    LLM returns structured JSON response with all 30 features.
    Each feature is either 0 or 1.
    """
    # 1. Define your schema structure (Assuming FEATURE_SCHEMA is defined globally)
    # If not, define it or pass it in.
    
    prompt = f"""
You are an expert in law-enforcement interaction detection.

You will analyze the following transcript and output ONLY a JSON object with 30 binary features (0 or 1).

Transcript:
{text}

Return JSON with this exact structure:
{FEATURE_SCHEMA}

Rules:
- Return 1 if the feature is present, 0 if absent with the feature as the looks 
- The json file should be a nested json file 
- No explanations. Only the JSON.
- Be strict: do not guess if unclear.
    """

    # 2. Define System Instructions as a SINGLE string (Fixes ValidationError)
    system_prompt_text = (
        'You are an expert in language model analysis. '
        'Return 1 if the feature is present, 0 if absent. '
        'No explanations. Only the JSON. '
        'Be strict: do not guess if unclear.'
    )

    # 3. Configure the model
    # response_mime_type="application/json" guarantees valid JSON output.
    # config = types.GenerateContentConfig(
    #     system_instruction=system_prompt_text,
    #     response_mime_type="application/json", 
    #     temperature=0.0 # Strict determinism
    # )
    
    # 4. Corrected API Call
    # 'contents' should be a simple string for single-turn, or a correctly structured list.
    # The previous error was caused by mixing OpenAI-style dicts with Gemini's schema.
    # print(prompt)
    response = client.models.generate_content(
          model='gemini-2.5-flash-lite', # Verify this Model ID is correct for your access level
          contents=prompt, 
        #   config=config,
    )

    # 5. robust Parsing
    content = response.text
    # print("----------content---------- ")
    # print(content)  
    # print("----------end content---------- ")
    
    # Clean markdown fences if they slip through (e.g., ```json ... ```)
    if content.startswith("```"):
        content = content.replace("```json", "").replace("```", "").strip()

    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        print("JSON parsing failed, raw output:")
        print(content)
        raise

    # 6. Flatten the schema
    # (Assumes data structure is Group -> Feature -> Value)
    flat = {}
    for group, features in data.items():
        if isinstance(features, dict):
            for f, val in features.items():
                flat[f] = int(val)
            # Calculate group scores
            flat[group + "_score"] = sum(int(v) for v in features.values())
        else:
            # Handle cases where schema might not be nested as expected
            print(f"Warning: Unexpected structure for group {group}")

    return flat

In [ ]:
import pandas as pd
import tqdm
import os
import time

# Assuming extract_llm_gemini_features is defined elsewhere
# from your_module import extract_llm_gemini_features 

def process_transcripts_csv(path, text_column="Transcript"): 
    # 1. Load the Source Data
    df = pd.read_csv(path, encoding='cp1252')
    
    if text_column not in df.columns:
        print(f"Error: Column '{text_column}' not found.")
        return None

    output_path = "./merge-output/tiktok_video_features.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # ======================================================
    # RESUME LOGIC: Load existing results to see what's done
    # ======================================================
    processed_indices = set()
    
    if os.path.exists(output_path):
        try:
            # Read the save file
            existing_df = pd.read_csv(output_path)
            
            # Check for 'original_index' to know which rows are done
            if 'original_index' in existing_df.columns:
                processed_indices = set(existing_df['original_index'].unique())
                print(f"🔄 Resuming: Found {len(processed_indices)} rows already processed in '{output_path}'.")
            else:
                print("⚠️ Save file found, but missing 'original_index'. Cannot resume safely. Starting fresh.")
        except Exception as e:
            print(f"⚠️ Could not load save file (starting fresh): {e}")

    # ======================================================
    
    results_buffer = []
    print(f"Processing remaining {len(df) - len(processed_indices)} rows...")

    try:
        # Loop through the dataframe
        for i, row in tqdm.tqdm(df.iterrows(), total=len(df)):
            
            # RESUME CHECK: If this index is already in our save file, SKIP IT
            if i in processed_indices:
                continue

            # 2. Extract Text
            text = row.get(text_column, "")
            
            # Skip empty text
            if pd.isna(text) or str(text).strip() == "":
                continue

            # 3. Call LLM
            try:
                time.sleep(6)
                features = extract_llm_gemini_features(str(text))
                
                # CRITICAL: Save the index so we can resume later if needed
                features["row_id"] = row.get('id', i)
                features["original_index"] = i 
                
                results_buffer.append(features)

                # 4. Save Incrementally (Append Mode)
                # Saving every 1 row is safest. Change to 10 or 50 for speed if desired.
                if len(results_buffer) >= 1:
                    new_rows_df = pd.DataFrame(results_buffer)
                    
                    # Check if file exists to determine if we need to write headers
                    file_exists = os.path.exists(output_path)
                    
                    # 'a' mode appends to the file instead of overwriting
                    new_rows_df.to_csv(output_path, mode='a', header=not file_exists, index=False)
                    
                    results_buffer = [] # Clear buffer

            except Exception as e:
                print(f"Error on row {i}: {e}")
                continue

    except KeyboardInterrupt:
        print("\n🛑 Process stopped by user.")
    
    finally:
        # Save any leftovers in buffer
        if results_buffer:
            new_rows_df = pd.DataFrame(results_buffer)
            file_exists = os.path.exists(output_path)
            new_rows_df.to_csv(output_path, mode='a', header=not file_exists, index=False)
        
        print("✅ Processing complete.")
        
        # Return the FULL dataset (Old + New loaded from file)
        if os.path.exists(output_path):
            return pd.read_csv(output_path)
        return pd.DataFrame()

# Usage
df_result = process_transcripts_csv("./input/all_videos_tiktok.csv", text_column="transcript")